# Local-Evidence SemEval Segmentation

Load a trusted classification artifact, rescore SemEval windows, and derive MG/HG regions and boundaries.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from notebooks.utils.cross_dataset_classification_utils import concat_or_empty, label_result_frame
from notebooks.utils.local_evidence_artifact_utils import load_local_evidence_artifact
from notebooks.utils.local_evidence_comparison_utils import expand_model_scores_to_tokens
from notebooks.utils.local_evidence_evaluation_utils import evaluate_local_evidence_semeval
from notebooks.utils.token_candidate_similarity_heatmaps_utils import load_token_labels

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
RUN_ID = "mixed-bbc-gemma-3-1b-v1"
EXPECTED_ARTIFACT_METADATA = {
    "scorer_llm": "gemma_3_1b_it", "training_regime": "Mixed+BBC",
    "training_sources": ["Kaggle", "MAGE", "Outfox", "BBC"], "cohort_seed": 42, "max_per_split": 500,
    "classifier_names": ["logistic_regression"], "window_lengths": [32, 64, 128], "stride": 16, "top_k": 20,
}
SEMEVAL_DATASETS = ("semeval2024_task8_subtask_c_train", "semeval2024_task8_subtask_c_dev")
SEMEVAL_PREPARED_ROOT = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC"
INSPECT_DATASET, INSPECT_TEXT_ID, INSPECT_WINDOW_LENGTH = "semeval2024_task8_subtask_c_train", None, 64


In [ ]:
artifact = load_local_evidence_artifact(
    DATA_ROOT, RUN_ID, expected_metadata=EXPECTED_ARTIFACT_METADATA, trust_artifact=True,
)
model = artifact.model
display(artifact.path, artifact.metadata, artifact.training_metrics, artifact.evaluation_results["metrics"])

In [ ]:
region_frames, evaluation_frames, predicted_region_frames = [], [], []
boundary_frames, prediction_frames, exclusion_frames, scored_frames = [], [], [], []
for semeval_dataset in SEMEVAL_DATASETS:
    annotation_path = SEMEVAL_PREPARED_ROOT / (
        "SemEval2024_task8_C_train_MG.csv" if semeval_dataset.endswith("_train") else "SemEval2024_task8_C_dev_MG.csv"
    )
    try:
        result = evaluate_local_evidence_semeval(
            model, PROCESSED_ROOT, pd.read_csv(annotation_path), dataset=semeval_dataset, split="MG", sustained_length=3,
        )
        labels = {
            "llm": model.llm_name, "training_regime": EXPECTED_ARTIFACT_METADATA["training_regime"],
            "evaluation_regime": "SemEval", "evaluation_dataset": semeval_dataset,
        }
        region_frames.append(label_result_frame(result.summary, block="semeval_region", **labels))
        evaluation_frames.append(label_result_frame(result.evaluations, block="semeval_region_detail", **labels))
        predicted_region_frames.append(label_result_frame(result.predicted_regions, block="semeval_region_detail", **labels))
        boundary_frames.append(label_result_frame(result.boundary_summary, block="semeval_boundary", **labels))
        prediction_frames.append(label_result_frame(result.boundary_predictions, block="semeval_boundary", **labels))
        exclusion_frames.append(label_result_frame(result.boundary_exclusions, block="semeval_boundary", **labels))
        scored_frames.append(result.scored_windows.assign(evaluation_dataset=semeval_dataset))
    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
        exclusion_frames.append(pd.DataFrame([{"evaluation_dataset": semeval_dataset, "reason": str(error)}]))

semeval_region_summary = concat_or_empty(region_frames)
semeval_region_evaluations = concat_or_empty(evaluation_frames)
semeval_predicted_regions = concat_or_empty(predicted_region_frames)
semeval_boundary_summary = concat_or_empty(boundary_frames)
boundary_predictions = concat_or_empty(prediction_frames)
boundary_exclusions = concat_or_empty(exclusion_frames)
semeval_scored_windows = concat_or_empty(scored_frames)
display(semeval_region_summary, semeval_boundary_summary, boundary_exclusions)

In [ ]:
if INSPECT_TEXT_ID is not None:
    selected = semeval_scored_windows.loc[
        semeval_scored_windows["evaluation_dataset"].eq(INSPECT_DATASET)
        & semeval_scored_windows["text_id"].eq(INSPECT_TEXT_ID)
        & semeval_scored_windows["window_length"].eq(INSPECT_WINDOW_LENGTH)
    ]
    token_count = len(load_token_labels(PROCESSED_ROOT, INSPECT_DATASET, model.llm_name, "MG")[INSPECT_TEXT_ID])
    token_scores = expand_model_scores_to_tokens(selected, token_count)
    plt.figure(figsize=(14, 3))
    plt.plot(token_scores["token_position"], token_scores["mg_score"])
    plt.ylim(0, 1)
    plt.xlabel("Token position")
    plt.ylabel("MG score")
    plt.title(f"{INSPECT_DATASET} {INSPECT_TEXT_ID}: token-level MG score")
    plt.show()
else:
    display(semeval_scored_windows.groupby(["evaluation_dataset", "model", "window_length"]).size().rename("scored_windows").reset_index())